# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadBilalFarooq/Assignment1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import duckdb, os, numpy as np, pandas as pd
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

RANDOM_SEED = 42

url = "https://raw.githubusercontent.com/MuhammadBilalFarooq/Assignment1/main/work/notebooks/w03_features.parquet"
features = pd.read_parquet(url)
features['ctr_early'] = np.where(features['imp_early'] > 0, features['clk_early'] / features['imp_early'], 0)

HF_TOKEN = os.environ.get('HF_TOKEN') or userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

content_attrs = con.sql(f"""
    SELECT
        content_hash_id, content_type, word_count, char_count, backlinks,
        search_volume, competition, main_intent, category_count,
        DATE_DIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days
    FROM read_parquet('{REL}/dim_content.parquet')
""").df()

features_v2 = features.merge(content_attrs, on='content_hash_id', how='left')
numeric_cols = ['word_count', 'char_count', 'backlinks', 'search_volume', 'competition']
for col in numeric_cols:
    features_v2[f'{col}_missing'] = features_v2[col].isnull().astype(int)
    features_v2[col] = features_v2[col].fillna(features_v2[col].median())
features_v2['main_intent'] = features_v2['main_intent'].fillna('unknown')
features_v2 = pd.get_dummies(features_v2, columns=['content_type', 'main_intent'], drop_first=True)

exclude_cols = ['client_hash_id', 'content_hash_id', 'imp_late', 'is_declining']
feature_cols_v2 = [c for c in features_v2.columns if c not in exclude_cols]
X2 = features_v2[feature_cols_v2]
y2 = features_v2['is_declining']
groups2 = features_v2['client_hash_id']

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx2, test_idx2 = next(gss2.split(X2, y2, groups=groups2))
X_train2, X_test2 = X2.iloc[train_idx2], X2.iloc[test_idx2]
y_train2, y_test2 = y2.iloc[train_idx2], y2.iloc[test_idx2]

rf = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=20,
    class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1
)
rf.fit(X_train2, y_train2)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y_true)[order[:k]].mean()

print("Setup complete.")
print(f"Feature table: {features_v2.shape}, Test set: {len(X_test2)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Setup complete.
Feature table: (92548, 26), Test set: 22531


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue below is scored by the Random Forest model (trained in Week 5, re-fit here on the same
grouped train split) across the full 92,548-item feature table. Each item gets a risk score
(predicted probability of decline), an action tier, and a reason code built from the model's top
contributing signals for that specific row -- so a reviewer sees not just "flagged" but "flagged
because of X."

Action tiers are set using the precision@K bands measured and reported in w05/w06: precision holds
up well through roughly the top 20-50 ranked items per client-scale slice, then degrades toward the
base rate beyond that (P@100 = 0.60 vs base rate 0.37, per w05 Section 3). Tiers reflect that: the
top band is where the model's confidence is best supported by evidence, and confidence honestly
drops further down the queue.
**Result:** 20 items landed in REVIEW_FIRST (top-20, matching the measured P@20=0.650 band), 80 in
REVIEW_SOON (top 21-100, where precision degrades toward the base rate), and the remaining 92,448 in
MONITOR_ONLY. The top-ranked items are dominated by content_age_days and char_count signals per the
model's global feature importances, with WEAK_POSITION as the most common individual reason code at
the very top of the queue.

In [9]:
# Score the FULL dataset (the actual playbook, not just the held-out test set)
full_scores = rf.predict_proba(X2)[:, 1]
features_v2['risk_score'] = full_scores

# Reason codes from the model's top global features, checked per-row
importances = pd.Series(rf.feature_importances_, index=feature_cols_v2).sort_values(ascending=False)
top_features = importances.head(5).index.tolist()
print(f"Top 5 features driving scores: {top_features}\n")

# Compute thresholds ONCE, outside the per-row function
pos_threshold_80 = features_v2['pos_early'].quantile(0.80)
char_threshold_25 = features_v2['char_count'].quantile(0.25)

def reason_code(row):
    reasons = []
    if 82 <= row['content_age_days'] <= 130:
        reasons.append('AGE_RISK_WINDOW')
    if row['clk_early'] == 0 and row['imp_early'] >= 20:
        reasons.append('ZERO_CLICKS')
    if row['pos_early'] >= pos_threshold_80:
        reasons.append('WEAK_POSITION')
    if row['char_count'] < char_threshold_25:
        reasons.append('THIN_CONTENT')
    return reasons if reasons else ['LOW_SIGNAL']

features_v2['reason_codes'] = features_v2.apply(reason_code, axis=1)

# Rank and assign action tiers based on measured precision@K bands
ranked_queue = features_v2.sort_values('risk_score', ascending=False).reset_index(drop=True)
ranked_queue['rank'] = ranked_queue.index + 1

def action_tier(rank):
    if rank <= 20:
        return 'REVIEW_FIRST'
    elif rank <= 100:
        return 'REVIEW_SOON'
    else:
        return 'MONITOR_ONLY'

ranked_queue['action'] = ranked_queue['rank'].apply(action_tier)

print(ranked_queue['action'].value_counts())
print()
display_cols = ['rank', 'content_hash_id', 'action', 'reason_codes', 'risk_score']
ranked_queue[display_cols].head(10)

Top 5 features driving scores: ['content_age_days', 'char_count', 'ctr_early', 'pos_early', 'word_count']

action
MONITOR_ONLY    92448
REVIEW_SOON        80
REVIEW_FIRST       20
Name: count, dtype: int64



,rank,content_hash_id,action,reason_codes,risk_score
0,1,content_cbd0fdbc5c6a1ded,REVIEW_FIRST,[WEAK_POSITION],0.885979
1,2,content_77ee484ce4333606,REVIEW_FIRST,[WEAK_POSITION],0.884903
2,3,content_48e429fec313253b,REVIEW_FIRST,[WEAK_POSITION],0.882356
3,4,content_0b5d0ab50b14a4e0,REVIEW_FIRST,[WEAK_POSITION],0.881065
4,5,content_469ce16a71d8ef05,REVIEW_FIRST,[WEAK_POSITION],0.880015
5,6,content_f06572a7dd29e56f,REVIEW_FIRST,[WEAK_POSITION],0.879815
6,7,content_d67f98572493f71d,REVIEW_FIRST,[WEAK_POSITION],0.878698
7,8,content_224077481f7c7698,REVIEW_FIRST,[WEAK_POSITION],0.878593
8,9,content_fb2c1cef7122724e,REVIEW_FIRST,[WEAK_POSITION],0.877901
9,10,content_f6239cb784e03a74,REVIEW_FIRST,[WEAK_POSITION],0.877831


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** this queue is a decision-support prioritization tool for a content team deciding
which pages to review first for possible refresh. It is not a certified prediction and not an
automated action system. A human reviewer looks at REVIEW_FIRST and REVIEW_SOON items, checks the
reason codes, and decides whether to act -- the model narrows attention, it does not make the call.

**Where it stops being valid:**
- **Validated split, not proven generalization:** precision@K numbers (P@20=0.650, P@50=0.580,
  P@100=0.600) come from one grouped train/test split with 8 held-out test clients (w05/w06). This
  is a measured, out-of-sample result, not a guarantee it holds for every new client or every future
  month.
- **Client-memorization gap is real and was measured, not assumed:** w06 showed a ~30-point precision
  gap between a random split (0.88 at P@50) and the honest grouped split (0.58) on the exact same
  data. The lower, grouped number is what this playbook relies on -- but it is a reminder that any
  future retraining must keep the grouped split, or the reported precision will not hold.
- **Survivorship bias in the age signal:** content_age_days is the top feature, but w05's own audit
  found an inverted-U pattern explained by survivorship bias (fragile old pages likely already pruned
  before this snapshot). This means "old content" flags should be read as "old content that survived
  this long," not as a general claim that age causes decline.
- **Zero-click blind spot:** w05's error analysis found the model's most confident wrong cases were
  all zero-early-click, moderate-impression pages it flagged as declining but that were actually
  stable. REVIEW_FIRST items with a ZERO_CLICKS reason code deserve extra scrutiny for this reason.
- **Below rank ~100, do not trust the score alone:** precision at these ranks is close to the base
  rate (0.37 in the w05 test split), meaning the model's picks there are close to random. MONITOR_ONLY
  items should not be acted on based on risk_score alone.

This is decision-support language throughout, per the claim ladder: "these pages look worth
reviewing first, because..." -- never "this page will decline" or "refreshing this page will fix it."

In [10]:
print("Section 2: intended use = decision-support prioritization, not automated action or certified prediction. Limits documented: split validity, measured client-memorization gap, survivorship bias in age signal, zero-click blind spot, precision floor below rank ~100.")


Section 2: intended use = decision-support prioritization, not automated action or certified prediction. Limits documented: split validity, measured client-memorization gap, survivorship bias in age signal, zero-click blind spot, precision floor below rank ~100.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any REVIEW_FIRST or REVIEW_SOON item, a human must check:**
1. **Read the reason codes and verify they make sense for this specific page** -- a WEAK_POSITION
   flag on a page that was intentionally deprioritized (e.g. merged, redirected, or replaced by a
   newer page) is a false positive, not a real decline.
2. **Check for a ZERO_CLICKS reason code specifically** -- per the documented blind spot in Section 2,
   these need extra scrutiny; a zero-click page may simply be stable and low-engagement, not declining.
3. **Confirm the page is still live and intentionally published** -- the model was trained on active
   content; a deleted, redirected, or deliberately retired page should be excluded regardless of score.
4. **Cross-check content_age_days against the survivorship caveat** -- an AGE_RISK_WINDOW flag alone
   is weaker evidence than AGE_RISK_WINDOW combined with another reason code.

**What should NEVER be automated:**
- **No automatic content edits, deletions, redirects, or republishing** based on risk_score or action
  tier alone. This queue identifies candidates for human review, not instructions to execute.
- **No automated client communication or reporting** claiming a page "will decline" or "will improve"
  -- the model has not

In [11]:
print("Section 3: human review checklist covers reason-code sanity check, zero-click scrutiny, live/published status, and age-flag corroboration. No-go list: no automated content changes, no automated client-facing claims, no action on MONITOR_ONLY tier, no cross-client generalization without re-validation.")


Section 3: human review checklist covers reason-code sanity check, zero-click scrutiny, live/published status, and age-flag corroboration. No-go list: no automated content changes, no automated client-facing claims, no action on MONITOR_ONLY tier, no cross-client generalization without re-validation.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Ongoing monitoring:**
- **Track realized precision@20 monthly** by checking, after 30-60 days, how many REVIEW_FIRST items
  from a given month's queue actually showed the decline pattern the model flagged. If realized
  precision drops meaningfully below the measured 0.650 baseline (e.g. below ~0.45-0.50 for two
  consecutive months), that is a signal the model's assumptions no longer match current portfolio
  behavior.
- **Track the base rate itself.** The

In [12]:
print("Section 4: monitoring = track realized precision@20 monthly against the 0.650 baseline, track base rate drift, periodically re-check the client-memorization gap from w06. Retrain triggers = major portfolio composition shift, new ground-truth month available, sustained precision drop, or warehouse schema changes.")

Section 4: monitoring = track realized precision@20 monthly against the 0.650 baseline, track base rate drift, periodically re-check the client-memorization gap from w06. Retrain triggers = major portfolio composition shift, new ground-truth month available, sustained precision drop, or warehouse schema changes.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Two things get exported: the full ranked queue as a CSV (regenerated on every run, not committed
per the repo's leak-guard policy), and a metrics JSON summarizing every key number from w04-w07
(committed to the repo as the "receipts" the paper's claims trace back to).

In [13]:
import json
import os

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# --- Export 1: the full ranked queue (regenerated, not committed) ---
export_cols = ['rank', 'client_hash_id', 'content_hash_id', 'action', 'reason_codes',
               'risk_score', 'imp_early', 'clk_early', 'pos_early', 'content_age_days', 'char_count']
ranked_queue[export_cols].to_csv('work/outputs/action_playbook_queue.csv', index=False)
print(f"Exported queue: {len(ranked_queue)} rows to work/outputs/action_playbook_queue.csv")

# --- Export 2: metrics JSON — the receipts for the paper (this DOES get committed) ---
metrics_summary = {
    "random_seed": RANDOM_SEED,
    "feature_table": {
        "rows": int(features_v2.shape[0]),
        "columns": int(features_v2.shape[1]),
        "source": "w03_features.parquet + dim_content.parquet"
    },
    "baseline_w04": {
        "rule": "weak_early_position (worst 20% of pos_early)",
        "base_rate": 0.286,
        "precision_at_20": 0.350,
        "precision_at_50": 0.300,
        "precision_at_100": 0.290
    },
    "model_w05": {
        "method": "RandomForestClassifier (200 trees, max_depth=8)",
        "test_base_rate": 0.370,
        "precision_at_20": 0.650,
        "precision_at_50": 0.580,
        "precision_at_100": 0.600,
        "top_features": top_features
    },
    "validation_w06": {
        "random_split_precision_at_50": 0.880,
        "grouped_split_precision_at_50": 0.580,
        "client_memorization_gap": 0.300,
        "leaky_feature_test_precision_at_20": 1.000,
        "honest_precision_at_20": 0.650
    },
    "playbook_w07": {
        "review_first_count": int((ranked_queue['action'] == 'REVIEW_FIRST').sum()),
        "review_soon_count": int((ranked_queue['action'] == 'REVIEW_SOON').sum()),
        "monitor_only_count": int((ranked_queue['action'] == 'MONITOR_ONLY').sum())
    }
}

with open('work/outputs/metrics_summary.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print("\nMetrics summary saved to work/outputs/metrics_summary.json")
print(json.dumps(metrics_summary, indent=2))

Exported queue: 92548 rows to work/outputs/action_playbook_queue.csv

Metrics summary saved to work/outputs/metrics_summary.json
{
  "random_seed": 42,
  "feature_table": {
    "rows": 92548,
    "columns": 28,
    "source": "w03_features.parquet + dim_content.parquet"
  },
  "baseline_w04": {
    "rule": "weak_early_position (worst 20% of pos_early)",
    "base_rate": 0.286,
    "precision_at_20": 0.35,
    "precision_at_50": 0.3,
    "precision_at_100": 0.29
  },
  "model_w05": {
    "method": "RandomForestClassifier (200 trees, max_depth=8)",
    "test_base_rate": 0.37,
    "precision_at_20": 0.65,
    "precision_at_50": 0.58,
    "precision_at_100": 0.6,
    "top_features": [
      "content_age_days",
      "char_count",
      "ctr_early",
      "pos_early",
      "word_count"
    ]
  },
  "validation_w06": {
    "random_split_precision_at_50": 0.88,
    "grouped_split_precision_at_50": 0.58,
    "client_memorization_gap": 0.3,
    "leaky_feature_test_precision_at_20": 1.0,
    "ho

Every section above is filled — ✅
Notebook runs top to bottom with no errors — do a final Runtime → Run all first
No client names, URLs, or private queries — ✅ (only hashed IDs)
Claims use careful words (observed, measured, directional, decision-support) — ✅ (Sections 2-4 use this consistently)
Committed to repo under work/notebooks/